In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100.0%
100.0%
100.0%
100.0%


In [3]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        x = self.flatten()
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Tensors

In [7]:
import torch
import numpy as np

In [8]:
data = [[1,2], [3,4]]
x_data = torch.tensor(data)

In [10]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
x_np

tensor([[1, 2],
        [3, 4]])

In [20]:
x_rand = torch.rand_like(x_data, dtype=torch.float)
x_rand

tensor([[0.9166, 0.1641],
        [0.3211, 0.0190]])

In [25]:
shape = (2, 3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.0404, 0.1120, 0.4739],
        [0.5965, 0.2091, 0.0624]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


In [26]:
data = [[0.0404, 2, 0.4739], [1, 0.2091, 0.0624]]
x_ten = torch.tensor(data)
print(x_ten.dtype)

torch.float32


In [27]:
# We move our tensor to the GPU if available
if torch.cuda.is_available():
  tensor = x_ten.to('cuda')
  print(f"Device tensor is stored on: {tensor.device}")

Device tensor is stored on: cuda:0


In [28]:
tensor = torch.ones(4, 4)

In [30]:
if torch.cuda.is_available():
  tensor = tensor.to('cuda')
  print(f"Device tensor is stored on: {tensor.device}")

Device tensor is stored on: cuda:0


In [32]:
torch.cuda.set_device(0)

In [33]:
tensor = torch.ones(4, 4)

In [37]:
t1 = torch.stack([tensor, tensor, tensor])

In [38]:
print(t1)

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])


In [41]:
t2 = tensor + tensor

In [42]:
t2 * t2

tensor([[4., 4., 4., 4.],
        [4., 4., 4., 4.],
        [4., 4., 4., 4.],
        [4., 4., 4., 4.]])

In [43]:
t2.matmul(t2)

tensor([[16., 16., 16., 16.],
        [16., 16., 16., 16.],
        [16., 16., 16., 16.],
        [16., 16., 16., 16.]])

In [44]:
t2 @ t2

tensor([[16., 16., 16., 16.],
        [16., 16., 16., 16.],
        [16., 16., 16., 16.],
        [16., 16., 16., 16.]])

In [45]:
t2.add_(5)

tensor([[7., 7., 7., 7.],
        [7., 7., 7., 7.],
        [7., 7., 7., 7.],
        [7., 7., 7., 7.]])

## Autograd

In [68]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

In [69]:
prediction = model(data)

In [70]:
loss = (prediction - labels).sum()
loss

tensor(-499.3326, grad_fn=<SumBackward0>)

In [71]:
loss.backward()

In [72]:
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
optim.step()

In [73]:
prediction = model(data)
loss = (prediction - labels).sum()
loss

tensor(-5036.8823, grad_fn=<SumBackward0>)

In [ ]:
for k in range(10):
    prediction = model(data)
    loss = (prediction - labels).sum()
    loss.backward()

    optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.01)
    optim.step()


ValueError: Invalid momentum value: -0.01

In [65]:
prediction = model(data)
loss = (prediction - labels).sum()
loss

tensor(-2.9244e+15, grad_fn=<SumBackward0>)

In [74]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = 3*a**3 - b**2

In [ ]:
external_grad = torch.tensor([1., 1.]) ## Need this - Remember - you will have to explicitly set the last tensor grad to 0
Q.backward(gradient=external_grad)

In [76]:
# check if collected gradients are correct
print(9*a**2 == a.grad)
print(-2*b == b.grad)

tensor([True, True])
tensor([True, True])


In [78]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False

In [79]:
model.fc = nn.Linear(512, 10) # Frozen layers except last

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()